# **GENETIC ALGORITHM**

This is a genetic algorithm implementation from scratch 

First, we need to see what is the correct solution for our problem, so we can compare it with the solution we get from our genetic algorithm. In this case, we wiil use the `linear regression` of scikit-learn to find the correct solution for our problem.

In [30]:
import random
import numpy as np
import pandas as pd

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


First, load the `.csv` file.

In [32]:
df = pd.read_csv("/content/drive/MyDrive/Student_Performance.csv")

In previous notebooks, we have seen that we can transform the `Extracurricular Activities` column into a binary column, where `1` means that the student has extracurricular activities and `0` means that the student does not have extracurricular activities. In this case, we will do the same thing.

In [33]:
df['Extracurricular Activities'] = df['Extracurricular Activities'].map({"Yes": 1, "No": 0})

Now, we need to split our dataset into X and y, where X is the features and y is the target variable. In this case, our target variable is the `Performance Index` column.

In [34]:
X = df.drop('Performance Index', axis=1)
Y = df['Performance Index']

## **LINEAR REGRESSION**

In [35]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.35, random_state=42)

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
lin_reg.intercept_, lin_reg.coef_

(np.float64(-33.78199549884449),
 array([2.85965187, 1.01555969, 0.59741919, 0.47323986, 0.18506513]))

## **GENETIC ALGORITHM**

### **STEP 1: INITIALIZATION**
In this step, we need to create a population of solutions. Each solution is represented as a grup of coefficients, which are the weights of the features in our linear regression model. We will create a population of 100 solutions, where each solution has 5 coefficients (one for each feature).

In [36]:
def initialize_poblation(size=100):
    random_seed = 42
    np.random.seed(random_seed)
    random.seed(42)
    poblacion = np.random.uniform(-3,3, size=(size, 5))
    poblacion = pd.DataFrame(poblacion)
    poblacion = poblacion.rename(columns={0: "theta_0", 1: "theta_1", 2: "theta_2", 3: "theta_3", 4: "theta_4"})
    poblacion["error"] = 0
    return poblacion

### **STEP 2: FITNESS EVALUATION**

In this step, we need to evaluate the fitness of each solution in our population. The fitness of a solution is a measure of how good the solution is. In our case, we will use the `mean squared error` as the fitness function. The lower the mean squared error, the better the solution.

In [37]:
def fitness( individuos: pd.DataFrame, X: pd.DataFrame, Y: pd.Series ):
    X = X.astype(float)
    Y = Y.astype(float).values
    resultados_error = []
    individuos = individuos.drop('error', axis=1)

    for _, individuo in individuos.iterrows():

        # Extraemos los genes
        theta = individuo.values.astype(float)

        y_pred = X.values @ theta
        error = np.mean((Y - y_pred) ** 2)
    
        resultados_error.append(error)
    
    error = np.array(resultados_error)
    individuos = individuos.copy()
    individuos["error"] = error

    return individuos


### **STEP 3: SELECTION**

We have a population of solutions and we have evaluated the fitness of each solution. Now, we need to select the best solutions to create the next generation. We will use the `elites` selection method, which means that we will select the top 50 solutions to create the next generation.

In [38]:
def best_50(poblacion: pd.DataFrame):
    ordered_poblation = poblacion.sort_values(by="error", ascending=True)
    top50= ordered_poblation.head(50)
    return top50

### **STEP 4: CROSSOVER**

In this step, we need to create new solutions by crossing over the selected solutions. We will a mutation probability or `gamma` to create new solutions. The probability mutation decides which coefficients will be mutated. IN this case or gamma is 0.6, which means that:

- If the random number is less than 0.6, we will cross the first and third coefficients of the two solutions.
- If the random number is greater than 0.6, we will cross the second and fourth coefficients of the two solutions.

In [39]:
def crossing(individuos: pd.DataFrame, gamma: float = 0.6):
  poblacion = individuos.copy()
  theta_0_values = individuos["theta_0"].values.flatten()
  theta_1_values = individuos["theta_1"].values.flatten()
  theta_2_values = individuos["theta_2"].values.flatten()
  theta_3_values = individuos["theta_3"].values.flatten()

  for i in range(0, len(theta_1_values),2):
    if random.random() <= gamma:
      theta_0_values[i], theta_0_values[i + 1] = theta_0_values[i + 1], theta_0_values[i]
      theta_2_values[i], theta_2_values[i + 1] = theta_2_values[i + 1], theta_2_values[i]
    else:
      theta_1_values[i], theta_1_values[i + 1] = theta_1_values[i + 1], theta_1_values[i]
      theta_3_values[i], theta_3_values[i + 1] = theta_3_values[i + 1], theta_3_values[i]

  poblacion["theta_0"] = theta_0_values
  poblacion["theta_1"] = theta_1_values
  poblacion["theta_2"] = theta_2_values
  poblacion["theta_3"] = theta_3_values

  nueva_poblacion = pd.concat([individuos, poblacion], ignore_index = False)
  nueva_poblacion["error"] = 0

  return nueva_poblacion

### **STEP 5: MUTATION**

Like in the crossover step, we will use the mutation probability or `gamma` to create new solutions. Perhaps, in this case the gamma probability determines if the coefficients will be mutated or not. If the random number is less than 0.65, we will mutate the coefficients by adding a random number between -0.001 and 0.001 to the coefficient.

In [40]:
def mutation(poblacion: pd.DataFrame, prob_mutacion: float = 0.65):
    poblacion = poblacion.drop('error', axis=1)
    for idx,_ in poblacion.iterrows():
        for col in poblacion.columns:
            if random.random() <= prob_mutacion:
                poblacion.loc[idx, col] += random.uniform(-0.001, 0.001)
    
    poblacion["error"] = 0
    return poblacion

### **STEP 5: IMPLEMENTATION**

Now, we have all the steps of our genetic algorithm, we need to implement it. We will create a function called `genetic_algorithm` that takes as input the population size, the number of generations, the features, and the target variable. The function will return the best solution found by the genetic algorithm.

In [41]:
def genetic_algorithm(generations: int, X: pd.DataFrame, Y: pd.Series, population_size: int = 100):
    poblation = initialize_poblation(size=population_size)
    poblation = fitness(
        X = X,
        Y = Y,
        individuos = poblation
    )

    for i in range(generations):
        top50 = best_50(poblacion= poblation)

        new_generation = crossing( 
            individuos= top50 
        )

        new_generation = mutation(
            poblacion = new_generation
        )

        poblation = new_generation

        poblation = fitness(
            X = X,
            Y = Y,
            individuos = poblation
        )
        
        # print(f'[INFO] Generación {i+1}')
    
    best_gen = poblation.sort_values(by='error').iloc[0]

    return best_gen

We will run the genetic algorithms in three different scenarios:

1. **Scenario 1**: We will run the genetic algorithm with a population size of 100 and a number of generations of 100.
2. **Scenario 2**: We will run the genetic algorithm with a population size of 200 and a number of generations of 500.
3. **Scenario 3**: We will run the genetic algorithm with a population size of 300 and a number of generations of 1000.

Finally, we will compare the solutions found by the genetic algorithm with the correct solution found by the linear regression model.

In [42]:
gen100  = genetic_algorithm(
    generations = 100,
    X = X,
    Y = Y
)

gen100

,16
theta_0,2.243885
theta_1,0.873213
theta_2,-1.038739
theta_3,-0.791667
theta_4,-1.113560
error,66.815746


In [43]:
gen500  = genetic_algorithm(
    generations = 500,
    X = X,
    Y = Y
)

gen500

,16
theta_0,2.167956
theta_1,0.884875
theta_2,-0.946322
theta_3,-0.745741
theta_4,-1.196568
error,72.741121


In [44]:
gen1000  = genetic_algorithm(
    generations = 1000,
    X = X,
    Y = Y
)

gen1000

,16
theta_0,2.278848
theta_1,0.826553
theta_2,-0.921094
theta_3,-0.794604
theta_4,-1.168983
error,44.973297


Pass the X to an numpy array.

In [45]:
X = X.to_numpy()

Now, let's compare the predictions by the genetic algorithm with the correct solution found by the linear regression model.

In [46]:
print(lin_reg.predict([X[2]]))

[45.1689919]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


To compare the predictions, we need drop the `error` column from the solutions found by the genetic algorithm.

In [47]:
gen100 = gen100.drop('error')
gen500 = gen500.drop('error')
gen1000 = gen1000.drop('error')

Let's predict!

In [48]:
print(X[2] @ gen100)
print(X[2] @ gen500)
print(X[2] @ gen1000)

53.67741196596458
53.91261142202071
51.563723254159754


In this case the best solution found by the genetic algorithm is the scenario 3 because it has the lowest mean squared error. However, the solution found by the linear regression model is better than the solution found by the genetic algorithm.

In [49]:
gen1000.to_numpy()

array([ 2.27884826,  0.8265534 , -0.92109432, -0.79460366, -1.16898315])

## **METRICS**

In this case, to evaluate our model, we need to create a `predict` function that takes the features and makes a prediction of the target variable.

In [50]:
def predict(X):
    gen = [ 2.27884826,  0.8265534 , -0.92109432, -0.79460366, -1.16898315]
    return X @ gen

In [51]:
y_pred_genetic = predict(X_test)
y_pred_genetic = y_pred_genetic.to_numpy()

In [52]:
y_pred = lin_reg.predict(X_test)

Let's calculate the following metrics:
- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- R-squared (R2)

In [53]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [54]:
mae_genetic_algorithm = mean_absolute_error(y_test, y_pred_genetic)
mse_genetic_algorithm = mean_squared_error(y_test, y_pred_genetic)
r2_genetic_algorithm = r2_score(y_test, y_pred_genetic)

print(f'Genetic Algorithm - MAE: {mae_genetic_algorithm}')
print(f'Genetic Algorithm - MSE: {mse_genetic_algorithm}')
print(f'Genetic Algorithm - R2 Score: {r2_genetic_algorithm}')

Genetic Algorithm - MAE: 5.559120229302858
Genetic Algorithm - MSE: 47.347228882002675
Genetic Algorithm - R2 Score: 0.8727104084271454


In [55]:
mae_linear_regression = mean_absolute_error(y_test, y_pred)
mse_linear_regression = mean_squared_error(y_test, y_pred)
r2_linear_regression = r2_score(y_test, y_pred)

print(f'Linear Regression - MAE: {mae_linear_regression}')
print(f'Linear Regression - MSE: {mse_linear_regression}')
print(f'Linear Regression - R2 Score: {r2_linear_regression}')

Linear Regression - MAE: 1.6172072555334853
Linear Regression - MSE: 4.123188763980212
Linear Regression - R2 Score: 0.9889151059916766
